In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
# write your code here

DB_PATH = "../db.sqlite3"
CSV_PATH = "joined_orders.csv"

sql = """
SELECT
    oi.id        AS order_item_id,
    oi.order_id  AS order_id,
    o.datetime   AS order_datetime,
    oi.product_id,
    p.name       AS product_name,
    p.price      AS product_price,
    oi.quantity  AS quantity
FROM restaurant_orderitem oi
JOIN restaurant_order o
    ON o.id = oi.order_id
JOIN restaurant_product p
    ON p.id = oi.product_id
;
"""

with sqlite3.connect(DB_PATH) as con:
    joined_df = pd.read_sql_query(sql, con)

assert joined_df.shape[0] == 74818, f"Expected 74818 rows, got {joined_df.shape[0]}"

joined_df.to_csv(CSV_PATH, index=False)

df = pd.read_csv(CSV_PATH)

assert df.shape[0] == 74818, f"Expected 74818 rows after CSV read, got {df.shape[0]}"

df.head()

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
# write your code here

top10 = (
    df.groupby("product_name", as_index=False)["quantity"]
      .sum()
      .sort_values("quantity", ascending=False)
      .head(10)
)

top10["percentage"] = (
    top10["quantity"] / top10["quantity"].sum() * 100
)

plt.figure(figsize=(9, 9))

plt.pie(
    top10["quantity"],
    labels=top10["product_name"],
    autopct="%.1f%%",
    startangle=140
)

plt.title("Top 10 Most Popular Products by Quantity Sold")
plt.tight_layout()
plt.show()


# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
# write your code here

df["item_price"] = df["product_price"] * df["quantity"]

top10_revenue = (
    df.groupby("product_name", as_index=False)["item_price"]
      .sum()
      .sort_values("item_price", ascending=False)
      .head(10)
)

top10_revenue["percentage"] = (
    top10_revenue["item_price"]
    / top10_revenue["item_price"].sum()
    * 100
)

plt.figure(figsize=(9, 9))

plt.pie(
    top10_revenue["item_price"],
    labels=top10_revenue["product_name"],
    autopct="%.1f%%",
    startangle=140
)

plt.title("Top 10 Products by Revenue Contribution")
plt.tight_layout()
plt.show()


# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
# write your code here
df["order_datetime"] = pd.to_datetime(df["order_datetime"])

df["order_hour"] = df["order_datetime"].dt.hour

df["item_price"] = df["product_price"] * df["quantity"]

income_by_hour = (
    df.groupby("order_hour", as_index=False)["item_price"]
      .sum()
      .sort_values("order_hour")
)

income_by_hour

plt.figure(figsize=(12, 6))

plt.bar(
    income_by_hour["order_hour"],
    income_by_hour["item_price"]
)

plt.xlabel("Order Hour (0–23)")
plt.ylabel("Total Restaurant Income")
plt.title("Total Restaurant Income by Order Hour")

plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()


# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
# write your code here

df["order_weekday"] = df["order_datetime"].dt.weekday
df["weekday_name"] = df["order_datetime"].dt.day_name()

df["item_price"] = df["product_price"] * df["quantity"]

weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

income_by_weekday = (
    df.groupby("weekday_name", as_index=False)["item_price"]
      .sum()
)

income_by_weekday["weekday_name"] = pd.Categorical(
    income_by_weekday["weekday_name"],
    categories=weekday_order,
    ordered=True
)

income_by_weekday = income_by_weekday.sort_values("weekday_name")
income_by_weekday

plt.figure(figsize=(12, 6))

plt.bar(
    income_by_weekday["weekday_name"],
    income_by_weekday["item_price"]
)

plt.xlabel("Day of the Week")
plt.ylabel("Total Restaurant Income")
plt.title("Total Restaurant Income by Day of the Week")

plt.tight_layout()
plt.show()